In [1]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('dataset/Telco-Customer-Churn.csv')
df.head(3)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [11]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors= 'coerce')
df = df.dropna(subset=['TotalCharges'])

In [13]:
X = df.iloc[: , 1:-1]
y = df.iloc[: , -1]

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

In [17]:
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler,LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [19]:
def conversion(x):
    if x == 1:
        return 'Y'
    return 'N'

X['SeniorCitizen'] = X['SeniorCitizen'].apply(conversion)

In [25]:
categorical_cols = list(X.select_dtypes(include=['object']).columns)
numerical_cols = list(X.select_dtypes(include=['int64','float64']).columns)
categorical_cols = categorical_cols + ['SeniorCitizen']

In [27]:
print("categocal-->",categorical_cols)
print("===============================================================")
print("numerical-->",numerical_cols)

categocal--> ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'SeniorCitizen']
numerical--> ['tenure', 'MonthlyCharges', 'TotalCharges']


In [37]:
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.2, random_state=21)

In [55]:
numerical_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("ohe", OneHotEncoder(handle_unknown = "ignore"))
])

preprocess = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000))
])

param_grid = [
    {
        "model": [LogisticRegression(max_iter=1000)],
        "model__C": [0.1, 1, 10]
    },
    
    {
        "model": [KNeighborsClassifier()],
        "model__n_neighbors": [1,3,5,7]
    },

    {
        "model": [RandomForestClassifier()],
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [5, 10]
        
    }
]


grid = GridSearchCV(pipeline, param_grid, cv=5 , n_jobs=-1)

grid.fit(X_train, y_train)

C:\ProgramData\anaconda3\Lib\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('numerical',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler())]),
                                                                         ['tenure',
                                                                          'MonthlyCharges',
                                                                          'TotalCharges']),
                                                                        ('categorical',
                                                                         Pipeline(steps=[('ohe',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['gender',
                                                                          'SeniorCitizen',
                                                                          'Partner',
                                                                          'Dependents',
                                                                          'PhoneService',
                                                                          'MultipleLine...
                                                                          'PaperlessBilling',
                                                                          'PaymentMethod',
                                                                          'SeniorCitizen'])])),
                                       ('model',
                                        LogisticRegression(max_iter=1000))]),
             n_jobs=-1,
             param_grid=[{'model': [LogisticRegression(max_iter=1000)],
                          'model__C': [0.1, 1, 10]},
                         {'model': [KNeighborsClassifier()],
                          'model__n_neighbors': [1, 3, 5, 7]},
                         {'model': [RandomForestClassifier()],
                          'model__max_depth': [5, 10],
                          'model__n_estimators': [100, 200, 300]}])

In [57]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)


Best Parameters: {'model': LogisticRegression(max_iter=1000), 'model__C': 0.1}
Best CV Score: 0.8055111111111112


In [59]:
print("Accuracy score is:", accuracy_score(y_test,y_pred))
print("Precision score is:", precision_score(y_test, y_pred, pos_label='Yes'))
print("Recall score is:", recall_score(y_test,y_pred, pos_label='Yes'))

Accuracy score is: 0.7903340440653873
Precision score is: 0.6791808873720137
Recall score is: 0.4975


In [61]:
import joblib

joblib.dump(best_model, 'cust_churn_model.pkl')

['cust_churn_model.pkl']